# 01 — Dhwani baseline inference (third-party checkpoint)

Smoke-tests the **third-party** ONNX model
[`ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model`](https://huggingface.co/ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model)
(MIT, pinned revision `ee01871`) on recordings you upload. We did not train this model.

**Read before interpreting output**
- Scores are softmax outputs of an *uncalibrated* classifier — **model scores, not probabilities**.
- The 0.5 threshold is a placeholder, not tuned on any validation set.
- Upstream code disagrees about preprocessing (evidence: `docs/UPSTREAM_AUDIT.md`). We do **not** pick one silently;
  every clip is scored under three controlled modes (all 16 kHz mono, same model, same clip):

  | Mode | Normalisation | Crop / windows | Source |
  |---|---|---|---|
  | A `train_peak` | peak → −3 dBFS | 4 s windows, 2 s hop | training notebook cell 4 (the run that exported this ONNX) |
  | B `train_zscore` | z-score | identical to A | normalisation used by the API / notebook inference cell |
  | C `api_replica` | z-score | first 3 s only, zero-padded to 4 s | deployed API `src/api/inference.py` |

  A vs B isolates normalisation; B vs C isolates crop/padding. The mode whose scores separate known genuine
  from known synthetic clips is *evidence* for the matching recipe — with 2 clips it is a hint, not proof.
- Label order (0 = genuine, 1 = synthetic) comes from upstream source and must be **confirmed on clips with known labels**.
- A handful of clips is a sanity check, not an accuracy estimate.

**Windowing policy (2–10 s clips)**
- < 2.0 s → rejected. > 10.0 s → only the first 10 s analysed (flagged `truncated`).
- ≤ 4 s → one window, zero-padded at the end (as in training).
- > 4 s → 4 s windows with 2 s hop; final window aligned to the clip end so every sample is covered.
- Clip score = mean of window scores (max also reported).
- Empty, unreadable, NaN, silent (peak < −50 dBFS) or mostly silent (< 1 s active signal) inputs are rejected — never replaced with zeros.

Run cells top to bottom. Runtime type: CPU is enough (ONNX Runtime uses `CPUExecutionProvider` here even on a GPU runtime).

In [12]:
# 1. Dependencies (Colab ships most of these; this is a no-op if already present)
!pip -q install onnxruntime librosa soundfile huggingface_hub

In [13]:
# 2. Settings
USE_DRIVE_CACHE = False   # True: keep the 1.26 GB model in Google Drive so new runtimes skip the download
DRIVE_CACHE_DIR = "/content/drive/MyDrive/vcd_cache"
MODES = ["train_peak", "train_zscore", "api_replica"]   # controlled comparison; see table above
UPLOAD_DIR = "/content/vcd_uploads"
RESULTS_DIR = "/content/vcd_results"

In [14]:
%%writefile dhwani_infer.py
"""Inference wrapper for the third-party Dhwani ONNX checkpoint.

Model: ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model (MIT),
file best_model.onnx at HF revision MODEL_REVISION. We did NOT train it.

Preprocessing is reproduced from the upstream training notebook
(github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API @ efd512c,
training/training_notebook.ipynb cell 4), which is the notebook whose cell 14
exports best_model.onnx with a (1, 64000) dummy input. See
docs/UPSTREAM_AUDIT.md for why we do not use the model card's z-score recipe
as the default.

Scores are softmax outputs of an UNCALIBRATED classifier. They are model
scores, not probabilities that a clip is synthetic.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np

# ---- pinned provenance -----------------------------------------------------
MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"
MODEL_REVISION = "ee0187142973e1f02050e47d2a31c55ee0263065"
MODEL_SHA256 = "d1c232bf4d7990526804b375c15804c0ee4e9b566478b908c5a57431491b7342"
MODEL_BYTES = 1262760439

# ---- preprocessing / windowing policy ---------------------------------------
SR = 16000
WIN = 64000            # 4.0 s: training crop length and ONNX export shape
HOP = 32000            # 2.0 s: same hop as the upstream API's sliding window
MIN_DUR_S = 2.0        # brief: 2-10 s clips
MAX_DUR_S = 10.0       # longer clips: analyse the first 10 s, flagged
PEAK_TARGET_DBFS = -3.0

# ---- rejection gates (input validation only; never fed to the model) --------
SILENT_PEAK_DBFS = -50.0   # whole clip quieter than this -> silent
ACTIVE_FRAME_S = 0.025
ACTIVE_REL_DB = 35.0       # frame counts as active if within 35 dB of loudest frame
ACTIVE_ABS_DBFS = -55.0    # ...and above this absolute level
MIN_ACTIVE_S = 1.0

# Class order from upstream dataset.py: labels = [0]*real + [1]*fake.
# Documented in source; must still be confirmed empirically on known clips.
LABELS = {0: "genuine", 1: "synthetic"}
SYNTHETIC_INDEX = 1
DECISION_THRESHOLD = 0.5   # placeholder, NOT tuned on any validation set

# Controlled comparison (docs/UPSTREAM_AUDIT.md):
#   train_peak   - training recipe: peak -3 dBFS, 4 s windows             (A)
#   train_zscore - identical windows, z-score instead of peak norm        (A vs B isolates normalisation)
#   api_replica  - deployed API: first 3 s, z-score, zero-pad to 4 s     (B vs C isolates crop/padding)
PREPROCESSING_MODES = ("train_peak", "train_zscore", "api_replica")


class AudioRejected(Exception):
    """Input cannot be analysed (unreadable, empty, silent, too short)."""


def sha256_file(path: str, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _dbfs(x: float) -> float:
    return float(20 * np.log10(max(x, 1e-12)))


# ---- loading ------------------------------------------------------------------
def load_audio(path: str):
    """Return (samples [channels, n] float32 at native rate, native_sr).

    Tries soundfile first (wav/flac/ogg), then librosa/audioread (mp3, m4a, opus
    via ffmpeg). Never substitutes zeros on failure.
    """
    import soundfile as sf

    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)  # (n, ch)
        y = y.T
    except Exception as sf_err:
        try:
            import librosa

            y, sr = librosa.load(path, sr=None, mono=False)
            y = np.atleast_2d(y).astype(np.float32)
        except Exception as lr_err:
            raise AudioRejected(
                f"unreadable audio (soundfile: {type(sf_err).__name__}; "
                f"librosa: {type(lr_err).__name__}: {lr_err})"
            ) from lr_err
    if y.size == 0 or y.shape[-1] == 0:
        raise AudioRejected("empty audio (0 samples)")
    if not np.all(np.isfinite(y)):
        raise AudioRejected("audio contains NaN/Inf samples")
    return y, int(sr)


def to_mono_16k(y: np.ndarray, sr: int) -> np.ndarray:
    mono = y.mean(axis=0) if y.shape[0] > 1 else y[0]
    if sr != SR:
        import librosa

        mono = librosa.resample(mono, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return mono.astype(np.float32)


def signal_stats(x: np.ndarray) -> dict:
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) if x.size else 0.0
    flen = int(ACTIVE_FRAME_S * SR)
    nfr = len(x) // flen
    if nfr:
        fr = x[: nfr * flen].reshape(nfr, flen).astype(np.float64)
        fdb = 10 * np.log10(np.mean(fr**2, axis=1) + 1e-20)
        active = (fdb > fdb.max() - ACTIVE_REL_DB) & (fdb > ACTIVE_ABS_DBFS)
        active_s = float(active.sum() * ACTIVE_FRAME_S)
    else:
        active_s = 0.0
    return {
        "peak_dbfs": round(_dbfs(peak), 2),
        "rms_dbfs": round(_dbfs(rms), 2),
        "active_s": round(active_s, 3),
        "clipping_frac": round(float(np.mean(np.abs(x) >= 0.999)) if x.size else 0.0, 5),
    }


def validate(x16: np.ndarray) -> tuple[np.ndarray, dict]:
    """Apply the 2-10 s policy and silence gates. Returns (analysed audio, info)."""
    dur = len(x16) / SR
    info = {"duration_s": round(dur, 3), "truncated": False}
    if dur < MIN_DUR_S:
        raise AudioRejected(f"too short: {dur:.2f} s < {MIN_DUR_S} s minimum")
    if dur > MAX_DUR_S:
        x16 = x16[: int(MAX_DUR_S * SR)]
        info["truncated"] = True
    info["analysed_duration_s"] = round(len(x16) / SR, 3)
    st = signal_stats(x16)
    info.update(st)
    if st["peak_dbfs"] < SILENT_PEAK_DBFS:
        raise AudioRejected(f"silent: peak {st['peak_dbfs']} dBFS < {SILENT_PEAK_DBFS} dBFS")
    if st["active_s"] < MIN_ACTIVE_S:
        raise AudioRejected(
            f"mostly silent: {st['active_s']:.2f} s active signal < {MIN_ACTIVE_S} s"
        )
    return x16, info


# ---- preprocessing --------------------------------------------------------------
def window_starts(n: int) -> list[int]:
    """4 s windows, 2 s hop, last window aligned to the clip end (full coverage)."""
    if n <= WIN:
        return [0]
    starts = list(range(0, n - WIN + 1, HOP))
    if starts[-1] + WIN < n:
        starts.append(n - WIN)
    return starts


def make_windows(x16: np.ndarray, mode: str) -> tuple[np.ndarray, list[int]]:
    """Return (windows [k, WIN] float32, start sample indices)."""
    if mode in ("train_peak", "train_zscore"):
        if mode == "train_peak":
            # Training notebook: peak-normalise the clip to -3 dBFS, then crop/zero-pad
            # the END to 64000 samples. No mean/variance normalisation.
            peak = float(np.max(np.abs(x16)))
            x = x16 * ((10 ** (PEAK_TARGET_DBFS / 20)) / (peak + 1e-6))
        else:
            # Same windows; z-score over the analysed clip (before padding).
            x = (x16 - x16.mean()) / np.sqrt(x16.var() + 1e-5)
        starts = window_starts(len(x))
        wins = []
        for s in starts:
            w = x[s : s + WIN]
            if len(w) < WIN:
                w = np.pad(w, (0, WIN - len(w)))
            wins.append(w)
        return np.stack(wins).astype(np.float32), starts
    if mode == "api_replica":
        # Deployed upstream API (src/api/inference.py): first 48000 samples,
        # zero-pad to 48000, z-score, then zero-pad to 64000 in predict().
        # Kept only as a comparison; it differs from training.
        w = x16[:48000]
        w = np.pad(w, (0, 48000 - len(w)))
        w = (w - w.mean()) / np.sqrt(w.var() + 1e-5)
        w = np.pad(w, (0, WIN - len(w)))
        return w[None, :].astype(np.float32), [0]
    raise ValueError(f"unknown preprocessing mode {mode!r}; use {PREPROCESSING_MODES}")


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


# ---- model ------------------------------------------------------------------------
def load_session(model_path: str, verify_sha: bool = True):
    import onnxruntime as ort

    if verify_sha:
        got = sha256_file(model_path)
        if got != MODEL_SHA256:
            raise RuntimeError(f"model sha256 mismatch: {got} != pinned {MODEL_SHA256}")
    so = ort.SessionOptions()
    sess = ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess


def describe_session(sess) -> dict:
    import onnxruntime as ort

    return {
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "session_providers": sess.get_providers(),
        "inputs": [(i.name, i.shape, i.type) for i in sess.get_inputs()],
        "outputs": [(o.name, o.shape, o.type) for o in sess.get_outputs()],
    }


def interface_self_test(sess, seed: int = 0) -> dict:
    """Probe what the checkpoint actually accepts (metadata alone proves nothing)."""
    rng = np.random.default_rng(seed)
    name = sess.get_inputs()[0].name
    out = {}
    base = (rng.standard_normal(WIN) * 0.1).astype(np.float32)
    ref = sess.run(None, {name: base[None]})[0]
    out["len_64000_batch1"] = {"ok": True, "shape": list(ref.shape)}
    for n in (32000, 48000, 96000):
        try:
            r = sess.run(None, {name: base[:n][None] if n <= WIN else np.tile(base, 2)[:n][None]})[0]
            out[f"len_{n}_batch1"] = {"ok": True, "shape": list(r.shape)}
        except Exception as e:
            out[f"len_{n}_batch1"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    try:
        b = np.stack([base, base[::-1].copy()])
        r = sess.run(None, {name: b})[0]
        r1 = sess.run(None, {name: base[::-1].copy()[None]})[0]
        out["batch2"] = {
            "ok": True,
            "shape": list(r.shape),
            "max_abs_diff_vs_individual": float(
                max(np.abs(r[0] - ref[0]).max(), np.abs(r[1] - r1[0]).max())
            ) if r.shape[0] == 2 else None,
        }
    except Exception as e:
        out["batch2"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    # determinism
    again = sess.run(None, {name: base[None]})[0]
    out["deterministic"] = bool(np.allclose(again, ref))
    return out


@dataclass
class Result:
    file: str
    status: str = "ok"
    reject_reason: str = ""
    file_sha256: str = ""
    native_sr: int | None = None
    channels: int | None = None
    info: dict = field(default_factory=dict)
    preprocessing: str = ""
    window_starts_s: list = field(default_factory=list)
    window_logits: list = field(default_factory=list)
    window_scores: list = field(default_factory=list)
    clip_score_mean: float | None = None
    clip_score_max: float | None = None
    predicted_label: str = ""
    latency_ms_model: float | None = None
    latency_ms_total: float | None = None


def analyse(sess, path: str, mode: str = "train_peak") -> Result:
    t0 = time.perf_counter()
    res = Result(file=path, preprocessing=mode)
    try:
        res.file_sha256 = sha256_file(path)
        y, sr = load_audio(path)
        res.native_sr, res.channels = sr, int(y.shape[0])
        x16 = to_mono_16k(y, sr)
        x16, info = validate(x16)
        res.info = info
        wins, starts = make_windows(x16, mode)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res

    name = sess.get_inputs()[0].name
    logits = []
    tm = time.perf_counter()
    for w in wins:  # batch=1: the export shape; batch>1 is probed separately
        logits.append(sess.run(None, {name: w[None]})[0][0])
    res.latency_ms_model = round((time.perf_counter() - tm) * 1000, 1)
    logits = np.asarray(logits, dtype=np.float64)
    scores = softmax(logits)[:, SYNTHETIC_INDEX]
    res.window_starts_s = [round(s / SR, 3) for s in starts]
    res.window_logits = [[round(float(a), 4) for a in row] for row in logits]
    res.window_scores = [round(float(s), 4) for s in scores]
    res.clip_score_mean = round(float(scores.mean()), 4)
    res.clip_score_max = round(float(scores.max()), 4)
    res.predicted_label = LABELS[SYNTHETIC_INDEX] if res.clip_score_mean >= DECISION_THRESHOLD else LABELS[0]
    res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
    return res


def result_row(res: Result, **extra) -> dict:
    d = asdict(res)
    info = d.pop("info")
    for k in ("window_starts_s", "window_logits", "window_scores"):
        d[k] = json.dumps(d[k])
    row = {**extra, **d, **info}
    row.update(
        decision_threshold=DECISION_THRESHOLD,
        label_mapping=json.dumps(LABELS),
        model_repo=MODEL_REPO,
        model_revision=MODEL_REVISION,
        model_sha256=MODEL_SHA256,
    )
    return row

Overwriting dhwani_infer.py


In [15]:
# 3. Import helper module + runtime report
import importlib, sys, platform, os, json
import numpy as np
sys.path.insert(0, os.getcwd())
import dhwani_infer as dh
importlib.reload(dh)

import onnxruntime as ort, librosa, soundfile
print("python", platform.python_version(), "| onnxruntime", ort.__version__,
      "| librosa", librosa.__version__, "| soundfile", soundfile.__version__)
try:
    import torch
    print("PyTorch CUDA available:", torch.cuda.is_available(),
          "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except ImportError:
    print("PyTorch not installed")
print("ONNX Runtime providers installed:", ort.get_available_providers())
print("NOTE: PyTorch CUDA availability does not mean ONNX Runtime uses the GPU. This notebook uses CPUExecutionProvider.")

python 3.13.16 | onnxruntime 1.31.0 | librosa 0.11.0 | soundfile 0.14.0
PyTorch CUDA available: True | GPU: Tesla T4
ONNX Runtime providers installed: ['AzureExecutionProvider', 'CPUExecutionProvider']
NOTE: PyTorch CUDA availability does not mean ONNX Runtime uses the GPU. This notebook uses CPUExecutionProvider.


In [16]:
# 4. Get the REAL pinned Dhwani checkpoint (downloads only if not already cached), verify sha256
from huggingface_hub import hf_hub_download

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    MODEL_PATH = os.path.join(DRIVE_CACHE_DIR, dh.MODEL_FILE)
    if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) == dh.MODEL_BYTES:
        print("Using Drive-cached model:", MODEL_PATH)
    else:
        print("Downloading", dh.MODEL_REPO, "@", dh.MODEL_REVISION[:7], "(1.26 GB) to Drive ...")
        MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION, local_dir=DRIVE_CACHE_DIR)
else:
    # Default HF cache: reuses a copy already downloaded earlier in this runtime
    # (e.g. by "AI Voice Detection.ipynb") when the blob matches the pinned revision.
    MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION)
    print("Model path:", MODEL_PATH)
assert os.path.getsize(MODEL_PATH) == dh.MODEL_BYTES, "unexpected model size"
marker = os.path.join(os.path.dirname(os.path.realpath(MODEL_PATH)), ".dhwani_sha256_ok")

stamp = f"{os.path.getsize(MODEL_PATH)}:{os.path.getmtime(MODEL_PATH)}"
if os.path.exists(marker) and open(marker).read() == stamp:
    print("sha256 previously verified for this file")
else:
    got = dh.sha256_file(MODEL_PATH)
    assert got == dh.MODEL_SHA256, f"sha256 mismatch: {got}"
    open(marker, "w").write(stamp)
    print("sha256 verified:", got)

Model path: /root/.cache/huggingface/hub/models--ayush2635--Dhwani-Multilingual-Deepfake-Audio-Detection-Model/snapshots/ee0187142973e1f02050e47d2a31c55ee0263065/best_model.onnx
sha256 previously verified for this file


In [17]:
# 5. Load session, print interface, probe what it really accepts
sess = dh.load_session(MODEL_PATH, verify_sha=False)  # sha checked in the previous cell
desc = dh.describe_session(sess)
for k, v in desc.items():
    print(f"{k}: {v}")
probe = dh.interface_self_test(sess)
print(json.dumps(probe, indent=2))

onnxruntime_version: 1.31.0
available_providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
session_providers: ['CPUExecutionProvider']
inputs: [('input', ['batch_size', 'time'], 'tensor(float)')]
outputs: [('output', [1, 2], 'tensor(float)')]
{
  "len_64000_batch1": {
    "ok": true,
    "shape": [
      1,
      2
    ]
  },
  "len_32000_batch1": {
    "ok": true,
    "shape": [
      1,
      2
    ]
  },
  "len_48000_batch1": {
    "ok": true,
    "shape": [
      1,
      2
    ]
  },
  "len_96000_batch1": {
    "ok": true,
    "shape": [
      1,
      2
    ]
  },
  "batch2": {
    "ok": true,
    "shape": [
      1,
      2
    ],
    "max_abs_diff_vs_individual": null
  },
  "deterministic": true
}


In [18]:
# 5b. Fetch ONE pre-registered English synthetic clip + its metadata from MLAAD (gated, CC-BY-NC-4.0)
import os, io, csv, json, random, shutil, hashlib, glob
import soundfile as sf
from huggingface_hub import hf_hub_download
from google.colab import userdata

MLAAD_REPO, MLAAD_REV = "mueller91/MLAAD", "30c3dec763fa4803f11c2df1557d3b0828395366"
MLAAD_GENERATOR, MLAAD_LANG, MLAAD_SEED = "Chatterbox", "en", 20261024
UPLOAD_DIR = globals().get("UPLOAD_DIR", "/content/vcd_uploads")
META_DIR = "/content/vcd_meta"
os.makedirs(UPLOAD_DIR, exist_ok=True); os.makedirs(META_DIR, exist_ok=True)

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access") from e

def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()

# 1) metadata for this generator only
folder = f"fake/{MLAAD_LANG}/{MLAAD_GENERATOR}"
meta_path = hf_hub_download(MLAAD_REPO, f"{folder}/meta.csv", repo_type="dataset",
                            revision=MLAAD_REV, token=HF_TOKEN)
text = open(meta_path, encoding="utf-8").read()
dialect = csv.Sniffer().sniff(text.splitlines()[0], delimiters="|,;\t")
rows_m = list(csv.DictReader(io.StringIO(text), dialect=dialect))
cols = list(rows_m[0].keys())
print(len(rows_m), "metadata rows; columns:", cols)
path_key = next((k for k in cols if k.strip().lower() in ("path", "file", "filename", "audio")), None)
dur_key = next((k for k in cols if k.strip().lower() in ("duration", "duration_s")), None)
assert path_key, f"no path column found in {cols}"

# 2) pre-registered order: sort by path, seeded shuffle; first 5-10 s clip (no detector involved)
cands = sorted(rows_m, key=lambda r: r[path_key])
random.Random(MLAAD_SEED).shuffle(cands)
checked, chosen = [], None
for r in cands:
    if dur_key and r[dur_key]:
        if not (5.0 <= float(r[dur_key]) <= 10.0):
            continue                      # filter on published duration, no download
    elif len(checked) >= 10:
        break                             # no duration column: cap trial downloads
    rel = f"{folder}/{os.path.basename(r[path_key].strip())}"
    local = hf_hub_download(MLAAD_REPO, rel, repo_type="dataset", revision=MLAAD_REV, token=HF_TOKEN)
    info = sf.info(local)
    checked.append((rel, round(info.duration, 3)))
    if 5.0 <= info.duration <= 10.0:
        chosen = (r, rel, local, info); break
assert chosen, f"no 5-10 s clip found; checked: {checked}"

# 3) byte copy into the upload dir (no re-encoding) + provenance sidecar
r, rel, local, info = chosen
sample_id, ext = os.path.splitext(os.path.basename(rel))
out_name = f"synthetic__{MLAAD_LANG}__{MLAAD_GENERATOR}__{sample_id}{ext}"
shutil.copyfile(local, os.path.join(UPLOAD_DIR, out_name))
sidecar = {
    "file_name": out_name, "label": "synthetic", "language": MLAAD_LANG,
    "generator": MLAAD_GENERATOR, "sample_id": sample_id,
    "dataset": MLAAD_REPO, "dataset_revision": MLAAD_REV, "license": "CC-BY-NC-4.0",
    "repo_path": rel, "sha256": _sha256(local),
    "selection_rule": f"meta.csv sorted by '{path_key}', random.Random({MLAAD_SEED}).shuffle, first clip 5-10 s",
    "clips_downloaded": checked,
    "audio": {"samplerate": info.samplerate, "channels": info.channels,
              "duration_s": round(info.duration, 3), "format": info.format, "subtype": info.subtype},
    "meta_csv_row": r,
}
with open(os.path.join(META_DIR, out_name + ".json"), "w", encoding="utf-8") as f:
    json.dump(sidecar, f, indent=2, ensure_ascii=False)
print(json.dumps(sidecar, indent=2, ensure_ascii=False))

paths = sorted(p for p in glob.glob(os.path.join(UPLOAD_DIR, "*")) if os.path.isfile(p))
print(len(paths), "file(s) in", UPLOAD_DIR, ":", [os.path.basename(p) for p in paths])

1000 metadata rows; columns: ['path', 'original_file', 'language', 'is_original_language', 'duration', 'training_data', 'model_name', 'architecture', 'transcript', 'speaker_similarity']
{
  "file_name": "synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav",
  "label": "synthetic",
  "language": "en",
  "generator": "Chatterbox",
  "sample_id": "emerald_city_of_oz_20_f000055",
  "dataset": "mueller91/MLAAD",
  "dataset_revision": "30c3dec763fa4803f11c2df1557d3b0828395366",
  "license": "CC-BY-NC-4.0",
  "repo_path": "fake/en/Chatterbox/emerald_city_of_oz_20_f000055.wav",
  "sha256": "9448a599a4d5cd420191182d20890592e69124c699f704903c22c3614f3fc0ac",
  "selection_rule": "meta.csv sorted by 'path', random.Random(20261024).shuffle, first clip 5-10 s",
  "clips_downloaded": [
    [
      "fake/en/Chatterbox/emerald_city_of_oz_20_f000055.wav",
      7.44
    ]
  ],
  "audio": {
    "samplerate": 22050,
    "channels": 1,
    "duration_s": 7.44,
    "format": "WAV",
    "subtype": "P

## Upload recordings

Name files so the ground truth is recorded automatically (anything after the prefix is free text):

- `genuine__<lang>__<note>.wav` — e.g. `genuine__hi__aman_phone_voicenote.m4a`
- `synthetic__<lang>__<generator>__<note>.wav` — e.g. `synthetic__en__elevenlabs__clone_of_harsh.mp3`

Files without a prefix are recorded as `unknown`. You can also fix labels in the `OVERRIDES` dict below.
Only upload audio you have the right to use; don't upload other people's voices without consent.

In [19]:
# 6. Upload (Colab file picker). Re-running adds more files.
import shutil, glob
os.makedirs(UPLOAD_DIR, exist_ok=True)
try:
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(name, os.path.join(UPLOAD_DIR, os.path.basename(name)))
except ImportError:
    print("Not in Colab: put files in", UPLOAD_DIR)
paths = sorted(p for p in glob.glob(os.path.join(UPLOAD_DIR, "*")) if os.path.isfile(p))
print(len(paths), "file(s):", [os.path.basename(p) for p in paths])

Saving genuine_hi_aman.m4a to genuine_hi_aman.m4a
2 file(s): ['genuine_hi_aman.m4a', 'synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav']


In [24]:
# 7. Metadata from filenames (+ optional manual overrides)
import pandas as pd

OVERRIDES = {
    # "some_file.wav": {"label": "genuine", "language": "hi", "generator": "", "note": ""},
    "genuine_hi_aman.m4a": {"label": "genuine", "language": "en"},
}

def parse_name(fn):
    parts = os.path.splitext(fn)[0].split("__")
    meta = {"label": "unknown", "language": "", "generator": "", "note": ""}
    if parts[0].lower() in ("genuine", "synthetic"):
        meta["label"] = parts[0].lower()
        if len(parts) > 1: meta["language"] = parts[1]
        if meta["label"] == "synthetic" and len(parts) > 2: meta["generator"] = parts[2]
        meta["note"] = "__".join(parts[3 if meta["label"] == "synthetic" else 2:])
    meta.update(OVERRIDES.get(fn, {}))
    return meta

meta = {os.path.basename(p): parse_name(os.path.basename(p)) for p in paths}
pd.DataFrame.from_dict(meta, orient="index")

,label,language,generator,note
genuine_hi_aman.m4a,genuine,en,,
synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav,synthetic,en,Chatterbox,emerald_city_of_oz_20_f000055


In [25]:
# 8. Inspect: validation report + playback (before any model scoring)
from IPython.display import Audio, display

for p in paths:
    fn = os.path.basename(p)
    print(f"--- {fn}  [{meta[fn]['label']}]")
    try:
        y, sr = dh.load_audio(p)
        x16 = dh.to_mono_16k(y, sr)
        print(f"native sr={sr} Hz, channels={y.shape[0]}, duration={y.shape[1]/sr:.2f} s", dh.signal_stats(x16))
        try:
            dh.validate(x16); print("validation: OK")
        except dh.AudioRejected as e:
            print("validation: REJECTED ->", e)
        display(Audio(x16, rate=dh.SR))  # playback is loudness-normalised by the browser widget
    except dh.AudioRejected as e:
        print("REJECTED ->", e)

--- genuine_hi_aman.m4a  [genuine]
native sr=48000 Hz, channels=1, duration=8.26 s {'peak_dbfs': -22.12, 'rms_dbfs': -39.15, 'active_s': 8.025, 'clipping_frac': 0.0}
validation: OK


/content/dhwani_infer.py:96: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=False)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


--- synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav  [synthetic]
native sr=22050 Hz, channels=1, duration=7.44 s {'peak_dbfs': -0.37, 'rms_dbfs': -20.68, 'active_s': 5.6, 'clipping_frac': 0.0}
validation: OK


In [26]:
# 9. Score every file with every preprocessing mode
rows = []
for p in paths:
    fn = os.path.basename(p)
    for mode in MODES:
        r = dh.analyse(sess, p, mode=mode)
        rows.append(dh.result_row(r, file_name=fn, ground_truth=meta[fn]["label"],
                                  language=meta[fn]["language"], generator=meta[fn]["generator"],
                                  note=meta[fn]["note"], provider=sess.get_providers()[0]))
df = pd.DataFrame(rows)
cols = ["file_name", "ground_truth", "preprocessing", "status", "reject_reason", "duration_s",
        "n_win", "window_scores", "clip_score_mean", "clip_score_max", "predicted_label",
        "latency_ms_model", "latency_ms_total"]
df["n_win"] = df["window_scores"].map(lambda s: len(json.loads(s)))
pd.set_option("display.max_colwidth", 80)
df[[c for c in cols if c in df]]

/content/dhwani_infer.py:96: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=False)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/dhwani_infer.py:96: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=False)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/dhwani_infer.py:96: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=False)
/usr/local/lib/python3.13/dist-packages/librosa/

,file_name,ground_truth,preprocessing,status,reject_reason,duration_s,n_win,window_scores,clip_score_mean,clip_score_max,predicted_label,latency_ms_model,latency_ms_total
0,genuine_hi_aman.m4a,genuine,train_peak,ok,,8.256,4,"[0.0001, 0.0001, 0.0001, 0.0001]",0.0001,0.0001,genuine,8695.3,8816.1
1,genuine_hi_aman.m4a,genuine,train_zscore,ok,,8.256,4,"[0.0001, 0.0001, 0.0001, 0.0001]",0.0001,0.0001,genuine,7382.4,7501.8
2,genuine_hi_aman.m4a,genuine,api_replica,ok,,8.256,1,[0.0001],0.0001,0.0001,genuine,1823.5,1955.1
3,synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav,synthetic,train_peak,ok,,7.440,3,"[0.9998, 0.9998, 0.9998]",0.9998,0.9998,synthetic,5574.0,5580.6
4,synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav,synthetic,train_zscore,ok,,7.440,3,"[0.9911, 0.9899, 0.9892]",0.9901,0.9911,synthetic,5633.8,5640.6
5,synthetic__en__Chatterbox__emerald_city_of_oz_20_f000055.wav,synthetic,api_replica,ok,,7.440,1,[0.9916],0.9916,0.9916,synthetic,1810.8,1817.2


In [28]:
# 10. Controlled preprocessing comparison + sanity summary (NOT an accuracy estimate)
ok = df[df.status == "ok"]
print("synthetic-class score = softmax(logits)[1]; uncalibrated; threshold 0.5 is a placeholder\n")
for mode, g in ok.groupby("preprocessing"):
    print(f"[{mode}]")
    by = g.groupby("ground_truth")["clip_score_mean"].agg(["count", "mean", "min", "max"])
    print(by.round(4).to_string())
    if {"genuine", "synthetic"} <= set(by.index):
        gm, sm = by.loc["genuine", "mean"], by.loc["synthetic", "mean"]
        if sm > gm:
            print("-> synthetic clips score higher than genuine on average: consistent with mapping 1 = synthetic")
        else:
            print("-> WARNING: genuine clips score >= synthetic. Mapping may be inverted, preprocessing wrong, or the model fails on these clips.")
    labelled = g[g.ground_truth.isin(["genuine", "synthetic"])]
    if len(labelled):
        agree = (labelled.predicted_label == labelled.ground_truth).sum()
        print(f"-> {agree}/{len(labelled)} labelled clips match at the placeholder threshold (tiny sample)")
    print()
print("Per-file clip_score_mean by mode (A vs B: normalisation; B vs C: crop):")
print(ok.pivot_table(index=["file_name", "ground_truth"], columns="preprocessing", values="clip_score_mean").round(4).to_string())
print()
print("rejected:", df[df.status != "ok"][["file_name", "preprocessing", "reject_reason"]].drop_duplicates("file_name").to_string(index=False) or "none")

synthetic-class score = softmax(logits)[1]; uncalibrated; threshold 0.5 is a placeholder

[api_replica]
              count    mean     min     max
ground_truth                               
genuine           1  0.0001  0.0001  0.0001
synthetic         1  0.9916  0.9916  0.9916
-> synthetic clips score higher than genuine on average: consistent with mapping 1 = synthetic
-> 2/2 labelled clips match at the placeholder threshold (tiny sample)

[train_peak]
              count    mean     min     max
ground_truth                               
genuine           1  0.0001  0.0001  0.0001
synthetic         1  0.9998  0.9998  0.9998
-> synthetic clips score higher than genuine on average: consistent with mapping 1 = synthetic
-> 2/2 labelled clips match at the placeholder threshold (tiny sample)

[train_zscore]
              count    mean     min     max
ground_truth                               
genuine           1  0.0001  0.0001  0.0001
synthetic         1  0.9901  0.9901  0.9901
-> syn

In [29]:
# 11. Save + download CSV (also copied to Drive if enabled)
import datetime
os.makedirs(RESULTS_DIR, exist_ok=True)
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_path = os.path.join(RESULTS_DIR, f"dhwani_predictions_{ts}.csv")
df.assign(run_utc=ts, onnxruntime=ort.__version__).to_csv(csv_path, index=False)
with open(os.path.join(RESULTS_DIR, f"dhwani_interface_{ts}.json"), "w") as f:
    json.dump({"session": desc, "probe": probe}, f, indent=2, default=str)
print("saved", csv_path)
if USE_DRIVE_CACHE:
    shutil.copy(csv_path, DRIVE_CACHE_DIR)
try:
    from google.colab import files
    files.download(csv_path)
except ImportError:
    pass

saved /content/vcd_results/dhwani_predictions_20261024T102407Z.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Report back
Paste into the chat: output of cells 3, 5 and 10, and attach the CSV. Mention how each clip was made
(device, app, generator, whether it went through WhatsApp/phone).